---
# IMPORTS

In [12]:
import sys, os
sys.path.insert(0, os.path.join('..'))   # project root on path

import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import wrds
import polars as pl
import pyarrow
import config

from src.data_loading import load_crsp, load_futures, load_crsp_polars, wrds_fetch, load_cz_monthly
from src.preprocessing import clean_crsp, clean_futures
from src.feature_engineering import add_target, add_volatility_momentum, crosssectional_rank, get_feature_cols 
from src.utils import generate_batches, train_val_test_split, split_batch, shrink

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.float_format', '{:.4f}'.format)

---
# Load daily dataset

In [6]:
data = pd.read_parquet(config.CRSP_PATH_CLEAN)
data = data.sort_values(['PERMNO', 'date']).reset_index(drop=True)
data['date'] = pd.to_datetime(data['date'])
print(data.shape)
print(data.columns.tolist())
print(f'Dataset RAM size before shrink : {data.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
data = shrink(data)
print(f'Dataset RAM size after shrink : {data.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
print(data.dtypes)
data.head()

(19603583, 4)
['PERMNO', 'date', 'ret', 'mkt_ret']
Dataset RAM size before shrink : 0.51Gb
Dataset RAM size after shrink : 0.37Gb
PERMNO              int32
date       datetime64[ms]
ret               float32
mkt_ret           float32
dtype: object


,PERMNO,date,ret,mkt_ret
0,10001,2000-01-03,0.0074,-0.0095
1,10001,2000-01-04,-0.0146,-0.0383
2,10001,2000-01-05,0.0148,0.0019
3,10001,2000-01-06,-0.0073,0.0010
4,10001,2000-01-07,-0.0074,0.0271


---
# Load Chen-Zimmerman Dataset

In [18]:
cz_daily = load_cz_monthly()
print(f'Dataset RAM size before shrink : {cz_daily.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
cz_daily = shrink(cz_daily)
print(f'Dataset RAM size after shrink : {cz_daily.memory_usage(index=True).sum() / 1024**3:.2f}Gb')

# Index cz dataset
cz_indexed = cz_daily.set_index('date').sort_index()
cz_indexed.index = pd.to_datetime(cz_indexed.index).astype('datetime64[ms]') # Prepare to merge

cz_indexed.to_parquet(config.CZ_PATH_CLEAN, compression='zstd')
cz_indexed.head()

Initial Size of the dataset: (1140, 206)
Total Date range : 1926-01-30 00:00:00 -> 2020-12-31 00:00:00
Dataset shape after dropping column with less than 90.0% completion: (1140, 57)
Total column dropped so far : 149
Dataset shape after dropping highly correlated (corr_coef > 0.95) columns: (1140, 55)
Total column dropped so far : 151
Dataset RAM size before shrink : 0.01Gb
Dataset RAM size after shrink : 0.01Gb


---
# Features Engineering on the daily dataset

In [7]:
from src.feature_engineering import build_features

"""

build_features function creates the following :

- 1 day reversal
- Momentum on multiple windows  
- Momentum scaled by volatility on multiple windows
- Volatility on multiple windows

"""

df = build_features(data, config.VALUE_RETURN, trading_interval=False)
 
df.head()

c:\Users\dario\Documents\Academique\EPFL\Master\MA-2\Machine Learning for Finance\ML-For-Finance-Project-LeoWunderli-329314-DarioFrey-344524\.venv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\dario\Documents\Academique\EPFL\Master\MA-2\Machine Learning for Finance\ML-For-Finance-Project-LeoWunderli-329314-DarioFrey-344524\.venv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


,PERMNO,date,ret,mkt_ret,reversal_1d,mom_scaled_5d,mom_5d,vol_5d,mom_scaled_21d,mom_21d,...,mom_scaled_63d,mom_63d,vol_63d,mom_scaled_126d,mom_126d,vol_126d,mom_scaled_252d,mom_252d,vol_252d,target
201,10001,2000-10-18,-0.0071,-0.0058,0.0478,-0.1288,-0.0149,-0.2534,0.2205,0.2225,...,0.3624,0.3400,-0.2376,0.2460,0.2415,-0.2669,0.1693,0.1433,-0.3210,0.0000
202,10001,2000-10-19,0.0000,0.0347,-0.0238,0.2008,0.1983,-0.4209,0.3127,0.3083,...,0.3661,0.3450,-0.2372,0.2283,0.2256,-0.2663,0.1663,0.1418,-0.3216,0.0071
203,10001,2000-10-20,0.0071,0.0059,-0.1657,-0.0070,-0.0096,-0.4258,0.2253,0.2244,...,0.3036,0.2897,-0.2471,0.1929,0.1921,-0.2678,0.1536,0.1306,-0.3226,-0.0071
204,10001,2000-10-23,-0.0071,-0.0008,0.0797,0.1743,0.0773,-0.4107,0.2316,0.2256,...,0.3008,0.2812,-0.2474,0.1935,0.1894,-0.2682,0.1575,0.1342,-0.3244,0.0143
205,10001,2000-10-24,0.0143,0.0017,-0.1373,-0.3645,-0.1235,-0.4532,0.1132,0.1310,...,0.3197,0.2889,-0.2476,0.2450,0.2379,-0.2690,0.1488,0.1236,-0.3256,-0.0141


In [22]:
"""

Run a check on the correlation between features
Optional to drop the one with too high correlation as they may not bring more relevant information

"""

corr_matrix = df.corr().abs()
print(corr_matrix)

                 PERMNO   date    ret  mkt_ret  reversal_1d  mom_scaled_5d  \
PERMNO           1.0000 0.0631 0.0005   0.0016       0.0046         0.0057   
date             0.0631 1.0000 0.0038   0.0217       0.0002         0.0002   
ret              0.0005 0.0038 1.0000   0.2743       0.0452         0.0309   
mkt_ret          0.0016 0.0217 0.2743   1.0000       0.0000         0.0000   
reversal_1d      0.0046 0.0002 0.0452   0.0000       1.0000         0.3806   
mom_scaled_5d    0.0057 0.0002 0.0309   0.0000       0.3806         1.0000   
mom_5d           0.0072 0.0002 0.0376   0.0000       0.3860         0.8970   
vol_5d           0.0435 0.0002 0.0076   0.0000       0.0142         0.0535   
mom_scaled_21d   0.0090 0.0002 0.0222   0.0000       0.1892         0.4392   
mom_21d          0.0107 0.0002 0.0246   0.0000       0.1875         0.3929   
vol_21d          0.0635 0.0002 0.0069   0.0000       0.0242         0.0706   
mom_scaled_63d   0.0146 0.0002 0.0157   0.0000       0.1136     

In [23]:
print(f'Dataset RAM size before shrink : {df.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
df = shrink(df)
print(f'Dataset RAM size after shrink : {df.memory_usage(index=True).sum() / 1024**3:.2f}Gb')


df.to_parquet(config.FEATURES_PATH_CLEAN, compression='zstd')

Dataset RAM size before shrink : 1.63Gb
Dataset RAM size after shrink : 1.63Gb


---
#  Batch

Generating the random sampled batch and merging the other dataset on the batch. 

We merge other dataset with the daily dataset only at the batch level in order to largely decrease the RAM usage duriung the operation considering the size of the daily dataframe with around 6700 stocks.

In [24]:
features = pd.read_parquet(config.FEATURES_PATH_CLEAN)
cz = pd.read_parquet(config.CZ_PATH_CLEAN)
print(f'Number of unique PERMNO in dataset : {features['PERMNO'].nunique()}')

split_batch_dict = split_batch(features, 'PERMNO', 'date', df_to_join=[cz], batch_number=config.BATCH_NUMBER, batch_size=config.BATCH_SIZE, overlap=False)

Number of unique PERMNO in dataset : 6647
Total unique dates: 6089
  train: 2000-10-16 → 2017-09-25  (4262 dates, 70.0%)
  val  : 2017-09-26 → 2021-05-12  (913 dates, 15.0%)
  test : 2021-05-13 → 2024-12-30  (914 dates, 15.0%)
Total unique dates: 6088
  train: 2000-10-17 → 2017-09-25  (4261 dates, 70.0%)
  val  : 2017-09-26 → 2021-05-12  (913 dates, 15.0%)
  test : 2021-05-13 → 2024-12-30  (914 dates, 15.0%)
Total unique dates: 6089
  train: 2000-10-16 → 2017-09-25  (4262 dates, 70.0%)
  val  : 2017-09-26 → 2021-05-12  (913 dates, 15.0%)
  test : 2021-05-13 → 2024-12-30  (914 dates, 15.0%)
Total unique dates: 6091
  train: 2000-10-12 → 2017-09-22  (4263 dates, 70.0%)
  val  : 2017-09-25 → 2021-05-12  (914 dates, 15.0%)
  test : 2021-05-13 → 2024-12-30  (914 dates, 15.0%)
Total unique dates: 6090
  train: 2000-10-16 → 2017-09-26  (4263 dates, 70.0%)
  val  : 2017-09-27 → 2021-05-13  (913 dates, 15.0%)
  test : 2021-05-14 → 2024-12-31  (914 dates, 15.0%)


---
# Handling NaN due to merge/join

As we merged the different dataset, we use how='left' in order to use the daily dataset as the baseline. However, due to differences in data range among datasets, we will have to handle NaN after merging.

This is a Work In Progress !